In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
def import_summary_stats(filepath):
    sumStats = pd.read_csv(filepath, delim_whitespace=True)
    split = sumStats['SNP'].str.split(':', expand=True)
    sumStats.dropna(subset=['OR', 'STAT', 'P'], inplace=True)
    sumStats['A1'] = split[2]
    sumStats['A2'] = split[3]
    sumStats['ID'] = split[4]
    sumStats["CHR"] = sumStats["CHR"].astype(str)
    sumStats["BP"] = sumStats["BP"].astype(int)
    return sumStats

In [3]:
def import_imputation_stats(filepath):
    imputationStats = pd.read_table(filepath, sep = '|', low_memory=False)
    imputationStats["CHROM"] = imputationStats["CHROM"].astype(str)
    imputationStats["POS"] = imputationStats["POS"].astype(int)
    imputationStats = imputationStats.sort_values(by=['CHROM','POS'])
    imputationStats['ID'] = imputationStats['ID'].map(lambda x: '.' if x == '0' else x)
    return imputationStats

In [4]:
def import_geno_matrix(filepath):
    matrix = pd.read_table(filepath, sep = " ")
    pattern = '_[TCGA]+$'
    columns = matrix.axes[1]
    columns_fixed = columns.str.replace(pat=pattern, repl="")
    matrix.columns = columns_fixed
    return matrix

In [5]:
def import_covars(filepath):
    covars=pd.read_csv(filepath, sep = '\t').dropna(axis=1)
    covars['IID'] = covars['IID'].astype(int)
    covars['FID'] = covars['FID'].astype(int)
    return covars

In [6]:
def merge_summary_imputation(sumStats, imputationStats):
    merged_df_A1 = sumStats.merge(imputationStats, how="left", left_on=["BP", "A2", "A1", "ID"], right_on=["POS", "REF", "ALT", "ID"])
    merged_df_A1 = merged_df_A1.dropna()
    merged_df_A2 = sumStats.merge(imputationStats, how="left", left_on=["BP", "A1", "A2", "ID"], right_on=["POS", "REF", "ALT", "ID"])
    merged_df_A2 = merged_df_A2.dropna()
    merged_df = pd.concat([merged_df_A2, merged_df_A1])
    merged_df = merged_df.drop_duplicates(subset=["SNP"])
    merged_df = merged_df.sort_values(by=["CHR", "BP"])
    merged_df = merged_df.rename(columns={"NMISS": "N", "R2":"INFO"})
    return merged_df

In [7]:
def merge_matrix_sumimp(matrix, sumimp):
    transposed_df = sumimp[['SNP', 'CHROM', 'BP','REF', 'ALT', 'OR', 'SE', 'STAT', 'P', 'MAF']].transpose()
    transposed_df.columns = transposed_df.iloc[0]
    transposed_df = transposed_df.drop(transposed_df.index[0])
    matching_columns = list(set(matrix.columns) & set(transposed_df.columns))  # The matching columns
    non_matching_columns_a = list(set(matrix.columns) - set(transposed_df.columns))  # The unique columns from 'a'
    # Merge only on matching columns, keeping all rows from 'a' and adding the 9 rows from 'b'
    final_df = pd.concat([matrix, transposed_df[matching_columns]], ignore_index=True)
    # Fill missing values in 'a'-specific columns with 0
    final_df[non_matching_columns_a] = final_df[non_matching_columns_a].fillna(0)
    final_df['IID'] = final_df['IID'].astype(int)
    final_df['FID'] = final_df['FID'].astype(int)
    return final_df

In [8]:
def merge_matsumimp_covars(matsumimp, covars):
    df = matsumimp.merge(covars, how = 'left').fillna("0")
    return df

In [9]:
matrix = import_geno_matrix(f"/scratch/eleiterw/AppliedProject/results/mlFiles/additive.remission.topmed.rsq30.top1k.raw")

In [10]:
print(matrix.head)

<bound method NDFrame.head of      FID  IID  PAT  MAT  SEX  PHENOTYPE  1:1991867:A:G:rs6605078_A  \
0    424  424    0    0    0         -9                          0   
1    158  158    0    0    0         -9                          0   
2    296  296    0    0    0         -9                          0   
3    403  403    0    0    0         -9                          0   
4     23   23    0    0    0         -9                          1   
..   ...  ...  ...  ...  ...        ...                        ...   
494   92   92    0    0    0         -9                          0   
495  369  369    0    0    0         -9                          0   
496    2    2    0    0    0         -9                          0   
497  450  450    0    0    0         -9                          0   
498  387  387    0    0    0         -9                          0   

     1:2053372:G:T:rs3121827_G  1:2068268:AC:A:rs140974664_AC  \
0                            1                              0   

In [24]:
pattern = '_[TCGA]+$'
columns = matrix.axes[1]
columns_fixed = columns.str.replace(pat=pattern, repl="", regex = True)


Index(['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENOTYPE',
       '1:1991867:A:G:rs6605078', '1:2053372:G:T:rs3121827',
       '1:2068268:AC:A:rs140974664', '1:2092073:T:C:rs116303242',
       ...
       '22:20038244:T:C:rs9606221', '22:20559470:A:G:rs177421',
       '22:22326116:C:G:rs192157063', '22:23450920:T:C:rs150928074',
       '22:26841963:C:T:rs2051636', '22:30921723:A:G:rs71331269',
       '22:36647323:T:C:rs111903107', '22:41258026:A:G:rs2229753',
       '22:43084507:T:C:rs5759151', '23:4680485:T:G:rs193139340'],
      dtype='object', length=947)
